#### 基因和小肽的密度在染色体上的分布对比

In [2]:
import pandas as pd
import numpy as np
import pyranges as pr

In [3]:
import sys
import os

def parse_attributes(attr_string):
    """解析 GFF3 第9列的属性为字典"""
    attrs = {}
    for item in attr_string.strip().split(';'):
        if '=' in item:
            key, value = item.split('=', 1)
            attrs[key] = value
    return attrs

def process_to_panno_format(input_file, output_file):
    print(f"正在处理文件: {input_file}")

    pep_counters = {}

    try:
        with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
            f_out.write("##gff-version 3\n")
            
            for line in f_in:
                line = line.strip()

                if not line or line.startswith("#"):
                    continue
                
                parts = line.split('\t')
                if len(parts) != 9:
                    continue
                    
                seqid, source, feature_type, start, end, score, strand, phase, attributes = parts

                if feature_type == 'CDS':
                    attr_dict = parse_attributes(attributes)

                    identity_str = attr_dict.get('Identity', '0')
                    try:
                        identity_val = float(identity_str)
                    except ValueError:
                        identity_val = 0.0
                    if seqid not in pep_counters:
                        pep_counters[seqid] = 0
                    pep_counters[seqid] += 1
                    new_id = f"pep_{seqid}_{pep_counters[seqid]}"
                    new_source = "minprot"
                    new_type = "Peptides"
                    new_score = f"{identity_val:.4f}"

                    new_strand = "." 
                    new_phase = "0"

                    new_attrs = f"ID={new_id};conf={identity_val:.2f}"

                    new_line = f"{seqid}\t{new_source}\t{new_type}\t{start}\t{end}\t{new_score}\t{new_strand}\t{new_phase}\t{new_attrs}\n"
                    
                    f_out.write(new_line)
                    
        print(f"处理完成！输出文件已保存为: {output_file}")
        print(f"共处理了 {sum(pep_counters.values())} 条肽段记录。")

    except FileNotFoundError:
        print(f"错误: 找不到文件 {input_file}，请检查路径是否正确。")
    except Exception as e:
        print(f"发生错误: {e}")


In [8]:
# #给miniprot的生成标准的注释格式
# import os
# input_path = "/data/user_home/2023122004/FFTransformer/data/TAIR/TAIR.sps.gff3"

# output_path = input_path.replace(".gff3", ".std.gff3")

# # 如果文件名没有.gff3后缀，防止出错，直接追加后缀
# if output_path == input_path:
#     output_path = input_path + ".std.gff3"

# process_to_panno_format(input_path, output_path)

正在处理文件: /data/user_home/2023122004/FFTransformer/data/TAIR/TAIR.sps.gff3
处理完成！输出文件已保存为: /data/user_home/2023122004/FFTransformer/data/TAIR/TAIR.sps.std.gff3
共处理了 35315 条肽段记录。


In [4]:
TAIR_genome = "/data/user_home/2023122004/FFTransformer/data/TAIR/genome/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa"
TAIR_gff = "/data/user_home/2023122004/FFTransformer/data/TAIR/gff3/Arabidopsis_thaliana.TAIR10.62.gff3"
TAIR_sp_gff = "/data/user_home/2023122004/FFTransformer/data/TAIR/TAIR.sps.std.gff3"

rice_genome = "/data/user_home/2023122004/FFTransformer/data/Oryza/genome/Oryza_sativa.IRGSP-1.0.dna.toplevel.fa"
rice_gff = "/data/user_home/2023122004/FFTransformer/data/Oryza/genome/Oryza_sativa.IRGSP-1.0.62.chr.gff3"
rice_sp_gff = "/data/user_home/2023122004/FFTransformer/data/Oryza/Oryza.sps.std.gff3" 

MT_genome = "/data/user_home/2023122004/FFTransformer/data/MtSSPdb/genome/Medicago_truncatula.MedtrA17_4.0.dna.toplevel.fa"
MT_gff = "/data/user_home/2023122004/FFTransformer/data/MtSSPdb/gff3/Medicago_truncatula.MtrunA17r5.0_ANR.62.gff3"
MT_sp_gff = "/data/user_home/2023122004/FFTransformer/data/MtSSPdb/miniprot/MT_sps.std.gff3"

maize_genome = '/data/user_home/2023122004/FFTransformer/data/maize/genome/Zea_mays.Zm-B73-REFERENCE-NAM-5.0.dna.toplevel.fa'
maize_gff = '/data/user_home/2023122004/FFTransformer/data/maize/gff3/ZmB73.gff3'
maize_sp_gff = "/data/user_home/2023122004/FFTransformer/data/maize/maize.sps.std.gff3"



In [5]:
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
import os
import re

def normalize_chrom_name(name):
    return str(name).replace("Chr", "").replace("chr", "")

def parse_attributes(attr_str):
    attributes = {}
    for item in attr_str.split(';'):
        if '=' in item:
            key, value = item.strip().split('=', 1)
            attributes[key] = value
    return attributes.get('ID', attributes.get('Parent', 'Unknown_ID'))

def is_main_chromosome(chrom_name):
    """
    判断是否为主染色体。
    过滤掉 scaffold, contig, mt (线粒体), pt (叶绿体/质体)。
    """
    name = str(chrom_name).lower()
    if 'scaffold' in name or 'contig' in name:
        return False
    
    # 2. 过滤线粒体 (Mt) 和 叶绿体 (Pt)
    # 常见命名: chrMt, chrPt, Mt, Pt, mitochondria, chloroplast
    if 'mito' in name or 'chloro' in name:
        return False

    if re.search(r'(chr|_)?(mt|pt|m|c)$', name):
        return False
        
    return True

def extract_sequences(gff_path, fasta_path, output_csv):
    print(f"[-] 正在加载基因组 FASTA: {fasta_path} ...")

    genome_dict = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        norm_id = normalize_chrom_name(record.id)
        genome_dict[norm_id] = record.seq
        genome_dict[record.id] = record.seq
        
    print(f"    已加载 {len(genome_dict)} 条序列记录。")

    print(f"[-] 正在处理 GFF 文件: {gff_path} ...")

    try:
        df = pd.read_csv(gff_path, sep='\t', comment='#', header=None, 
                         names=['seqid', 'source', 'type', 'start', 'end', 'score', 'strand', 'phase', 'attributes'],
                         low_memory=False)
    except Exception as e:
        print(f"    [Error] 读取 GFF 失败: {e}")
        return

    extracted_data = []
    skipped_count = 0 

    for index, row in df.iterrows():
        chrom = str(row['seqid'])

        if not is_main_chromosome(chrom):
            skipped_count += 1
            continue

        norm_chrom = normalize_chrom_name(chrom)
        
        start = int(row['start']) - 1
        end = int(row['end'])
        strand = row['strand']
        feat_id = parse_attributes(str(row['attributes']))

        if norm_chrom in genome_dict:
            ref_seq = genome_dict[norm_chrom]
        elif chrom in genome_dict:
            ref_seq = genome_dict[chrom]
        else:
            print(f"    [Warning] 主染色体 {chrom} 未在 FASTA 中找到，跳过 {feat_id}")
            continue

        seq_slice = ref_seq[start:end]

        if strand == '-':
            final_seq = seq_slice.reverse_complement()
        else:
            final_seq = seq_slice
            
        seq_str = str(final_seq)
        
        extracted_data.append({
            'ID': feat_id,
            'Chrom': chrom,
            'Start': row['start'],
            'End': row['end'],
            'Strand': strand,
            'Length': len(seq_str),
            'Start_Codon': seq_str[:3],
            'Sequence': seq_str
        })

    result_df = pd.DataFrame(extracted_data)
    result_df.to_csv(output_csv, index=False)
    
    print(f"[-] 处理完成！")
    print(f"    已过滤非主染色体/Scaffold 记录: {skipped_count} 条")
    print(f"    共提取有效序列: {len(result_df)} 条")
    print(f"    结果已保存至: {output_csv}")

In [6]:
TAIR_sp = '/data/user_home/2023122004/FFTransformer/data/TAIR_sp.csv'
maize_sp = '/data/user_home/2023122004/FFTransformer/data/maize_sp.csv'
rice_sp = '/data/user_home/2023122004/FFTransformer/data/rice_sp.csv'
MT_sp = '/data/user_home/2023122004/FFTransformer/data/MT_sp.csv'

extract_sequences(TAIR_sp_gff, TAIR_genome, TAIR_sp)
extract_sequences(maize_sp_gff, maize_genome, maize_sp)
extract_sequences(rice_sp_gff, rice_genome, rice_sp)
extract_sequences(MT_sp_gff, MT_genome, MT_sp)

[-] 正在加载基因组 FASTA: /data/user_home/2023122004/FFTransformer/data/TAIR/genome/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa ...
    已加载 7 条序列记录。
[-] 正在处理 GFF 文件: /data/user_home/2023122004/FFTransformer/data/TAIR/TAIR.sps.std.gff3 ...
[-] 处理完成！
    已过滤非主染色体/Scaffold 记录: 859 条
    共提取有效序列: 34456 条
    结果已保存至: /data/user_home/2023122004/FFTransformer/data/TAIR_sp.csv
[-] 正在加载基因组 FASTA: /data/user_home/2023122004/FFTransformer/data/maize/genome/Zea_mays.Zm-B73-REFERENCE-NAM-5.0.dna.toplevel.fa ...
    已加载 685 条序列记录。
[-] 正在处理 GFF 文件: /data/user_home/2023122004/FFTransformer/data/maize/maize.sps.std.gff3 ...
[-] 处理完成！
    已过滤非主染色体/Scaffold 记录: 0 条
    共提取有效序列: 16558 条
    结果已保存至: /data/user_home/2023122004/FFTransformer/data/maize_sp.csv
[-] 正在加载基因组 FASTA: /data/user_home/2023122004/FFTransformer/data/Oryza/genome/Oryza_sativa.IRGSP-1.0.dna.toplevel.fa ...
    已加载 63 条序列记录。
[-] 正在处理 GFF 文件: /data/user_home/2023122004/FFTransformer/data/Oryza/Oryza.sps.std.gff3 ...
[-] 处理完成！
    已过滤非主染色体/Scaffo

In [28]:
import pandas as pd
from Bio import SeqIO
import math
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

species_config = {
    "Arabidopsis": {
        "genome": "/data/user_home/2023122004/FFTransformer/data/TAIR/genome/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa",
        "gene_gff": "/data/user_home/2023122004/FFTransformer/data/TAIR/gff3/Arabidopsis_thaliana.TAIR10.62.gff3",
        "sp_gff": "/data/user_home/2023122004/FFTransformer/data/TAIR/TAIR.sps.std.gff3"
    },
    "Rice": {
        "genome": "/data/user_home/2023122004/FFTransformer/data/Oryza/genome/Oryza_sativa.IRGSP-1.0.dna.toplevel.fa",
        "gene_gff": "/data/user_home/2023122004/FFTransformer/data/Oryza/genome/Oryza_sativa.IRGSP-1.0.62.chr.gff3",
        "sp_gff": "/data/user_home/2023122004/FFTransformer/data/Oryza/Oryza.sps.std.gff3"
    },
    "Medicago": {
        "genome": "/data/user_home/2023122004/FFTransformer/data/MtSSPdb/genome/Medicago_truncatula.MedtrA17_4.0.dna.toplevel.fa",
        "gene_gff": "/data/user_home/2023122004/FFTransformer/data/MtSSPdb/gff3/Medicago_truncatula.MtrunA17r5.0_ANR.62.gff3",
        "sp_gff": "/data/user_home/2023122004/FFTransformer/data/MtSSPdb/miniprot/MT_sps.std.gff3"
    },
    "Maize": {
        "genome": '/data/user_home/2023122004/FFTransformer/data/maize/genome/Zea_mays.Zm-B73-REFERENCE-NAM-5.0.dna.toplevel.fa',
        "gene_gff": '/data/user_home/2023122004/FFTransformer/data/maize/gff3/ZmB73.gff3',
        "sp_gff": "/data/user_home/2023122004/FFTransformer/data/maize/maize.sps.std.gff3"
    }
}

OUTPUT_FILE = "all_species_adaptive_density.csv"

def normalize_chrom_name(name):
    """标准化染色体名称，移除 Chr 前缀"""
    return str(name).replace("Chr", "").replace("chr", "")

def is_main_chromosome(chrom_name):
    """只保留纯数字的主染色体"""
    norm_name = normalize_chrom_name(chrom_name)
    if norm_name.isdigit():
        return True
    return False

def determine_window_size(chrom_lengths):
    """
    根据染色体平均长度，自适应选择更大的窗口大小，降低图形密度
    """
    if not chrom_lengths:
        return 1_000_000 

    avg_len = sum(chrom_lengths.values()) / len(chrom_lengths)

    if avg_len < 40_000_000:       # 平均 < 40MB (如拟南芥, 水稻)
        win_size = 500_000         # 从 200KB 放大到 500KB (如果还是密集，可以改为 1_000_000)
    elif avg_len < 100_000_000:    # 平均 < 100MB (如苜蓿)
        win_size = 2_000_000      # 从 500KB 放大到 1MB
    else:                          # 平均 > 100MB (如玉米)
        win_size = 5_000_000       # 从 1MB 放大到 2MB (玉米甚至可以尝试 5_000_000 也就是 5MB)
        
    print(f"    [Auto-Config] 平均染色体长度: {avg_len/1e6:.1f} Mb -> 推荐窗口: {win_size/1000:.0f} kb")
    return win_size

# ================= 3. 数据读取与处理 =================

def get_chrom_lengths(fasta_path):
    """读取 FASTA 获取主染色体长度"""
    print(f"  [FASTA] Reading {os.path.basename(fasta_path)} ...")
    chrom_lens = {}
    
    for record in SeqIO.parse(fasta_path, "fasta"):
        if is_main_chromosome(record.id):
            norm_id = normalize_chrom_name(record.id)
            chrom_lens[norm_id] = len(record.seq)

    sorted_keys = sorted(chrom_lens.keys(), key=lambda x: int(x))
    print(f"    -> 找到 {len(chrom_lens)} 条主染色体: {sorted_keys}")
    return chrom_lens

def load_feature_positions(gff_path, target_type='gene'):
    """读取 GFF 提取 Start 位置"""
    print(f"  [GFF] Parsing {os.path.basename(gff_path)} (Filter: {target_type if target_type else 'ALL'})...")
    positions = {}
    try:
        df = pd.read_csv(gff_path, sep='\t', comment='#', header=None, 
                         usecols=[0, 2, 3], names=['seqid', 'type', 'start'], 
                         low_memory=False)
        
        if target_type:
            df = df[df['type'] == target_type]
            
        for _, row in df.iterrows():
            chrom = str(row['seqid'])
            if not is_main_chromosome(chrom): continue
            
            norm_chrom = normalize_chrom_name(chrom)
            start = int(row['start'])
            
            if norm_chrom not in positions: positions[norm_chrom] = []
            positions[norm_chrom].append(start)
            
    except Exception as e:
        print(f"    [Error] GFF读取失败: {e}")
    return positions

def calculate_species_density(species_name, paths):
    results = []

    chrom_lengths = get_chrom_lengths(paths['genome'])
    
    if not chrom_lengths:
        print(f"    [Warning] {species_name} 没有找到有效的主染色体，跳过。")
        return []
    window_size = determine_window_size(chrom_lengths)
    gene_coords = load_feature_positions(paths['gene_gff'], target_type='gene')
    sp_coords = load_feature_positions(paths['sp_gff'], target_type=None) 
    
    print(f"  [Calc] Calculating density (Window: {window_size/1000:.0f}kb)...")
    
    sorted_chroms = sorted(chrom_lengths.keys(), key=lambda x: int(x))
    
    for chrom_norm in sorted_chroms:
        length = chrom_lengths[chrom_norm]
        num_windows = math.ceil(length / window_size)
        
        gene_counts = [0] * num_windows
        sp_counts = [0] * num_windows

        if chrom_norm in gene_coords:
            for pos in gene_coords[chrom_norm]:
                idx = (pos - 1) // window_size
                if 0 <= idx < num_windows: gene_counts[idx] += 1

        if chrom_norm in sp_coords:
            for pos in sp_coords[chrom_norm]:
                idx = (pos - 1) // window_size
                if 0 <= idx < num_windows: sp_counts[idx] += 1

        scaling_factor = 1_000_000 / window_size 
        
        for i in range(num_windows):
            g_count = gene_counts[i]
            s_count = sp_counts[i]
            
            results.append({
                'Species': species_name,
                'Chromosome': chrom_norm,
                'Window_Size_bp': window_size, 
                'Start_bp': i * window_size + 1,
                'End_bp': min((i + 1) * window_size, length),
                'Gene_Count': g_count, 
                'SP_Count': s_count,   
                'Gene_Density_perMb': g_count * scaling_factor, 
                'SP_Density_perMb': s_count * scaling_factor
            })
            
    return results

import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors


def plot_density_distribution(df, output_prefix="Density_Plot"):
    """绘制分布图：基因面积图+小肽热力图。超过10条染色体时自动分为两列绘制。"""
    print("\n[-] 正在绘制分布图...")
    sns.set(style="ticks", font_scale=1.0) 
    
    species_list = df['Species'].unique()

    colors = ['#cbe1ab', '#f6f5a1', '#fcd279', '#f39e5b', '#9b0024']
    cmap_custom = mcolors.LinearSegmentedColormap.from_list('pep_cmap', colors)
    
    for sp in species_list:
        sp_data = df[df['Species'] == sp]
        used_window = sp_data.iloc[0]['Window_Size_bp']
        win_label = f"{int(used_window/1000)}kb" if used_window < 1000000 else f"{used_window/1000000:.1f}Mb"
        
        chroms = sorted(sp_data['Chromosome'].unique(), key=lambda x: int(x))
        n_chroms = len(chroms)

        n_cols = 2 if n_chroms > 8 else 1
        n_rows = math.ceil(n_chroms / n_cols)

        fig_width = 8 if n_cols == 1 else 18
        fig = plt.figure(figsize=(fig_width, 2.5 * n_rows))

        outer_gs = gridspec.GridSpec(n_rows, n_cols, hspace=0.6, wspace=0.25) 
        
        print(f"  绘制 {sp} (Window: {win_label}, Layout: {n_rows} 行 x {n_cols} 列)...")

        axes_to_align = [[] for _ in range(n_cols)]
        
        for i, chrom in enumerate(chroms):
            data = sp_data[sp_data['Chromosome'] == chrom]
            
            x = data['Start_bp'] / 1_000_000
            y_gene = data['Gene_Count'] 
            y_pep = data['SP_Count'] 

            row_idx = i // n_cols  # 行索引 (例如: 0, 0, 1, 1, 2, 2)
            col_idx = i % n_cols   # 列索引 (例如: 0, 1, 0, 1, 0, 1)
            
            # 使用计算出的坐标 outer_gs[row_idx, col_idx] 来放置子图
            inner_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer_gs[row_idx, col_idx], 
                                                        height_ratios=[4, 1], hspace=0)
            
            ax_gene = fig.add_subplot(inner_gs[0])
            ax_pep  = fig.add_subplot(inner_gs[1], sharex=ax_gene)

            axes_to_align[col_idx].append(ax_gene)
            axes_to_align[col_idx].append(ax_pep)

            color_line = '#6e7f8b' 
            color_fill = '#D0E4EF' 
            
            ax_gene.plot(x, y_gene, color=color_line, linewidth=1.2)
            ax_gene.fill_between(x, 0, y_gene, color=color_fill, alpha=0.9)
            
            ax_gene.set_ylabel("Gene", fontsize=10, fontweight='bold', 
                               rotation=0, labelpad=15, va='center', ha='right')
            ax_gene.set_ylim(0, max(y_gene)*1.15 if max(y_gene) > 0 else 1)
            
            ax_gene.text(0.01, 0.85, f"Chr {chrom}", transform=ax_gene.transAxes,
                         fontsize=11, fontweight='bold', va='top', ha='left')
            
            ax_gene.spines['top'].set_visible(False)
            ax_gene.spines['right'].set_visible(False)
            ax_gene.spines['bottom'].set_visible(False)
            ax_gene.tick_params(axis='x', bottom=False, labelbottom=False)

            pep_array = y_pep.values.reshape(1, -1)
            window_size_mb = data.iloc[0]['Window_Size_bp'] / 1_000_000
            x_edges = list(x) + [x.iloc[-1] + window_size_mb]
            
            ax_pep.pcolormesh(x_edges, [0, 1], pep_array, cmap=cmap_custom, 
                              shading='flat', edgecolors='black', linewidth=0.1)
            
            ax_pep.set_ylabel("Pep", fontsize=10, fontweight='bold', 
                              rotation=0, labelpad=15, va='center', ha='right')
            ax_pep.set_yticks([]) 
            
            for spine in ax_pep.spines.values():
                spine.set_visible(True)
                spine.set_color('black')
                spine.set_linewidth(0.8)
            
            ax_gene.set_xlim(0, max(x_edges))

            is_bottom_of_col = (i + n_cols >= n_chroms)
            
            if is_bottom_of_col:
                ax_pep.set_xlabel("Chromosomal Position (Mb)", fontsize=11, labelpad=8)
            else:
                ax_pep.tick_params(axis='x', bottom=True, labelbottom=False)

        for col_axes in axes_to_align:
            if col_axes:
                fig.align_ylabels(col_axes)

        plt.tight_layout(pad=0) 
        filename = f"{output_prefix}_{sp}.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  -> 保存: {filename}")

if __name__ == "__main__":
    all_data = []

    for species, paths in species_config.items():
        print(f"\n 处理物种: {species}")
        try:
            species_data = calculate_species_density(species, paths)
            all_data.extend(species_data)
        except Exception as e:
            print(f"  [Error] 处理 {species} 时出错: {e}")
            import traceback
            traceback.print_exc()

    if all_data:
        print(f"\n {OUTPUT_FILE} ...")
        df_result = pd.DataFrame(all_data)

        print("\n 各物种使用的窗口大小:")
        print(df_result[['Species', 'Window_Size_bp']].drop_duplicates())
        
        df_result.to_csv(OUTPUT_FILE, index=False)
        plot_density_distribution(df_result)

    else:
        print("\n[Warning] 无数据生成。")

=== 开始处理 (自适应窗口模式) ===

[-] 处理物种: Arabidopsis
  [FASTA] Reading Arabidopsis_thaliana.TAIR10.dna.toplevel.fa ...
    -> 找到 5 条主染色体: ['1', '2', '3', '4', '5']
    [Auto-Config] 平均染色体长度: 23.8 Mb -> 推荐窗口: 500 kb
  [GFF] Parsing Arabidopsis_thaliana.TAIR10.62.gff3 (Filter: gene)...
  [GFF] Parsing TAIR.sps.std.gff3 (Filter: ALL)...
  [Calc] Calculating density (Window: 500kb)...

[-] 处理物种: Rice
  [FASTA] Reading Oryza_sativa.IRGSP-1.0.dna.toplevel.fa ...
    -> 找到 12 条主染色体: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
    [Auto-Config] 平均染色体长度: 31.1 Mb -> 推荐窗口: 500 kb
  [GFF] Parsing Oryza_sativa.IRGSP-1.0.62.chr.gff3 (Filter: gene)...
  [GFF] Parsing Oryza.sps.std.gff3 (Filter: ALL)...
  [Calc] Calculating density (Window: 500kb)...

[-] 处理物种: Medicago
  [FASTA] Reading Medicago_truncatula.MedtrA17_4.0.dna.toplevel.fa ...
    -> 找到 8 条主染色体: ['1', '2', '3', '4', '5', '6', '7', '8']
    [Auto-Config] 平均染色体长度: 48.1 Mb -> 推荐窗口: 2000 kb
  [GFF] Parsing Medicago_truncatula.Mtr

/tmp/ipykernel_31778/2013809383.py:300: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0)


  -> 保存: Density_Plot_Arabidopsis.png
  绘制 Rice (Window: 500kb, Layout: 6 行 x 2 列)...


/tmp/ipykernel_31778/2013809383.py:300: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0)


  -> 保存: Density_Plot_Rice.png
  绘制 Medicago (Window: 2.0Mb, Layout: 8 行 x 1 列)...


/tmp/ipykernel_31778/2013809383.py:300: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0)


  -> 保存: Density_Plot_Medicago.png
  绘制 Maize (Window: 5.0Mb, Layout: 5 行 x 2 列)...


/tmp/ipykernel_31778/2013809383.py:300: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0)


  -> 保存: Density_Plot_Maize.png

=== 全部完成！===
